# Clustering de clientes


### Perfiles de clientes BAJA+2 con Random Forest

Este notebook es una **adaptación en celdas** del script `z501_cluster_rf.py` provisto por la cátedra. El código no fue modificado: se lo dividió en celdas y se agregaron explicaciones en Markdown para facilitar su lectura y ejecución paso a paso.

### Qué hace el script?

Genera el archivo `clusters_tendencias.pdf` con perfiles de clientes que se dieron de baja (BAJA+2), usando un Random Forest para caracterizar hojas del bosque y KMeans para agrupar clientes según esos perfiles.

### Imports

Se importan las librerías necesarias. Se fuerza el backend `Agg` de matplotlib **antes** de importar `pyplot` porque el script no necesita mostrar gráficos en pantalla, sólo escribirlos a un PDF (esto es imprescindible si se corre en un entorno sin interfaz gráfica).

In [80]:
from __future__ import annotations

import argparse
import time
from pathlib import Path

import duckdb
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.backends.backend_pdf import PdfPages  # noqa: E402
from sklearn.cluster import KMeans  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


### Constantes globales

Nombres de columnas del dataset, colores para graficar cada cluster, paleta de tinta/grilla, y las tres formas posibles de calcular la "banda" alrededor de la tendencia mensual (`ic95`, `desvio`, `iqr`).

 `ID_COL`, `MES_COL`, `TARGET_COL`, `GRUPO_COL`, `CLUSTER_COL`: nombres de columnas usadas en todo el script.

  `RAIZ_COL`: marca las hojas de un árbol que no tuvieron ningún split (árbol de un solo nodo); esas filas no aportan información y se excluyen del clustering.
  
   `_T0` y `log(...)`: un cronómetro simple para loguear el progreso del script con el tiempo transcurrido desde que arrancó.

In [2]:
ID_COL = "numero_de_cliente"
MES_COL = "foto_mes"
TARGET_COL = "ternaria"
GRUPO_COL = "grupo"  # 1 = cliente BAJA+2, 0 = cliente fiel
CLUSTER_COL = "cluster"
NON_FEATURE_COLS = (ID_COL, MES_COL, TARGET_COL, GRUPO_COL)
RAIZ_COL = "_sin_split"  # hojas sin split (árbol de un solo nodo): no entran al clustering

COLORES_CLUSTER = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, INK_2, GRID = "#14181a", "#4b534e", "#d7dbd3"

BANDAS = {
    "ic95": ("media", "IC 95% de la media (± 1,96 · desvío / √n)"),
    "desvio": ("media", "± 1 desvío estándar"),
    "iqr": ("mediana", "Q25–Q75"),
}

_T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - _T0:6.1f}s] {msg}", flush=True)

## 1. Datos: todos los BAJA+2 con su historia + igual cantidad de fieles.

Esta sección arma la muestra sobre la que va a trabajar todo el resto del script, usando **DuckDB** para consultar el CSV directamente con SQL (sin cargarlo entero en memoria con pandas primero).

La consulta `_QUERY_MUESTRA` hace lo siguiente:

`raw`: lee el CSV completo, forzando el id de cliente a `BIGINT`.
`churn`: ids únicos que en algún mes tuvieron `clase_ternaria = 'BAJA+2'`.
`n_meses`: cantidad total de meses distintos en el dataset.
`fieles`: clientes que estuvieron presentes **todos** los meses y que nunca pasaron por `BAJA+1` ni `BAJA+2`.


Se ordena por un hash determinístico de `id + seed` y se recorta con `LIMIT` a la misma cantidad de clientes que hay en `churn` — esto arma un muestreo reproducible (mismo seed → misma muestra de fieles) y balanceado en cantidad de clientes con los BAJA+2.

Se unen ambos grupos (`grupo = 1` para BAJA+2, `grupo = 0` para fieles) y se ordena por id y mes.

`cargar_muestra(...)` ejecuta esa consulta y devuelve un DataFrame de pandas, forzando `clase_ternaria` a tipo string.
`columnas_features(...)` devuelve las columnas numéricas del DataFrame que no son columnas "administrativas" (id, mes, target, grupo) — es decir, las variables que va a usar el Random Forest.

In [3]:
_QUERY_MUESTRA = """
WITH raw AS (
    SELECT * REPLACE (CAST({id} AS BIGINT) AS {id})
    FROM read_csv('{csv}', sample_size = -1)
),
churn AS (
    SELECT DISTINCT {id} FROM raw WHERE {target} = 'BAJA+2'
),
n_meses AS (
    SELECT COUNT(DISTINCT {mes}) AS n FROM raw
),
-- fieles: presentes todos los meses y nunca BAJA+1/BAJA+2. ORDER BY hash(...) es un
-- shuffle determinístico por seed, así el LIMIT es un muestreo reproducible.
fieles AS (
    SELECT {id}
    FROM raw
    GROUP BY {id}
    HAVING COUNT(*) = (SELECT n FROM n_meses)
       AND SUM(CASE WHEN {target} IN ('BAJA+1', 'BAJA+2') THEN 1 ELSE 0 END) = 0
    ORDER BY hash({id} + {seed})
    LIMIT (SELECT COUNT(*) FROM churn)
)
SELECT raw.*, 1 AS {grupo} FROM raw JOIN churn USING ({id})
UNION ALL
SELECT raw.*, 0 AS {grupo} FROM raw JOIN fieles USING ({id})
ORDER BY {id}, {mes}
"""


def cargar_muestra(csv_path: Path, seed: int) -> pd.DataFrame:
    query = _QUERY_MUESTRA.format(
        csv=csv_path.as_posix(), id=ID_COL, mes=MES_COL, target=TARGET_COL, grupo=GRUPO_COL, seed=seed
    )
    df = duckdb.connect().execute(query).df()
    df[TARGET_COL] = df[TARGET_COL].astype("string")
    return df


def columnas_features(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in NON_FEATURE_COLS and pd.api.types.is_numeric_dtype(df[c])]

## 2. Random Forest sobre filas cliente-mes; target = "el cliente es BAJA+2"

`entrenar_rf(...)` entrena un `RandomForestClassifier` donde cada fila es una combinación cliente-mes y el target es la columna `grupo` (1 = BAJA+2, 0 = fiel).

Detalles importantes de los hiperparámetros:

`min_samples_leaf` alto (default 50): fuerza a que cada hoja represente una **población** de clientes-mes, no una fila memorizada individualmente. Esto es clave porque después se va a usar el atributo que definió cada hoja como una característica del cliente, y para que eso tenga sentido estadístico la hoja debe agrupar muchos casos.
`class_weight="balanced"`: compensa que un cliente fiel aporta 6 filas (una por mes que estuvo presente) mientras que un BAJA+2 aporta entre 2 y 5, evitando que el modelo se sesgue hacia la clase con más filas.
`oob_score=True`: permite estimar el accuracy "out-of-bag" sin necesitar un set de test aparte.`max_features="sqrt"`: cada split evalúa sólo una raíz cuadrada de los atributos, la configuración estándar de Random Forest para clasificación.

In [4]:
def entrenar_rf(X: np.ndarray, y: np.ndarray, n_trees: int, min_samples_leaf: int, seed: int) -> RandomForestClassifier:
    # min_samples_leaf alto: hojas que sean poblaciones de clientes, no filas memorizadas.
    # class_weight compensa que un fiel aporta 6 filas y un BAJA+2 entre 2 y 5.
    rf = RandomForestClassifier(
        n_estimators=n_trees, min_samples_leaf=min_samples_leaf, max_features="sqrt",
        class_weight="balanced", oob_score=True, n_jobs=-1, random_state=seed,
    )
    rf.fit(X, y)
    return rf

## 3. Atributo del último split de cada hoja + conteo por cliente

Esta es la parte conceptualmente más interesante del script: convierte el bosque entrenado en un **perfil por cliente**.

`atributo_de_hoja(estimator)`: para un árbol individual, devuelve para cada nodo el atributo con el que se hizo el split de su nodo **padre** (es decir, "qué atributo me trajo hasta acá").

Para la raíz del árbol no hay padre, así que queda en `-1`. Aplicado a una hoja, esto da "el atributo que definió esa hoja".

`contar_hojas_por_id(...)`: para cada fila (cliente-mes) y cada árbol del bosque, encuentra en qué hoja cayó (`rf.apply(X)`) y qué atributo definió esa hoja. Después agrupa por cliente (`id`) y cuenta cuántas veces (a lo largo de todos sus meses y todos los árboles) cada atributo fue el que definió la hoja en la que cayó. El resultado `C` es una matriz cliente × atributo de conteos; cada fila suma `n_meses(id) × n_trees`. Las hojas sin split (raíz) se acumulan en la columna especial `RAIZ_COL`. `normalizar(...)`: convierte esos conteos en proporciones por fila (cada cliente suma 1), descartando la columna de raíz y los atributos que nunca definieron ninguna hoja.

La intuición: si dos clientes son tratados de forma parecida por el bosque (caen en hojas definidas por los mismos atributos con frecuencias similares), tienen un perfil similar.

In [5]:
def atributo_de_hoja(estimator) -> np.ndarray:
    """Para cada nodo, el atributo con el que se hizo el último split antes de llegar
    a él (el atributo de su padre). En las hojas es "el atributo que define la hoja".
    -1 para la raíz."""
    t = estimator.tree_
    mapa = np.full(t.node_count, -1, dtype=np.int64)
    internos = np.flatnonzero(t.children_left != -1)
    mapa[t.children_left[internos]] = t.feature[internos]
    mapa[t.children_right[internos]] = t.feature[internos]
    return mapa


def contar_hojas_por_id(rf: RandomForestClassifier, X: np.ndarray, ids: np.ndarray, feature_names: list[str]) -> pd.DataFrame:
    """C[id, a] = cantidad de pares (mes, árbol) en que una fila del id cayó en una
    hoja cuyo último split fue el atributo a. Cada fila suma n_meses(id) × n_trees."""
    hojas = rf.apply(X)  # (n_filas, n_trees)
    n_feat = len(feature_names)
    codigos = np.empty_like(hojas)
    for j, est in enumerate(rf.estimators_):
        codigos[:, j] = atributo_de_hoja(est)[hojas[:, j]]
    codigos[codigos < 0] = n_feat

    idx, ids_unicos = pd.factorize(ids)
    C = np.zeros((len(ids_unicos), n_feat + 1), dtype=np.int64)
    np.add.at(C, (np.repeat(idx, hojas.shape[1]), codigos.ravel()), 1)
    return pd.DataFrame(C, index=pd.Index(ids_unicos, name=ID_COL), columns=feature_names + [RAIZ_COL])


def normalizar(C: pd.DataFrame) -> pd.DataFrame:
    """Proporciones por fila (cada cliente suma 1), sin atributos que nunca definieron una hoja."""
    P = C.drop(columns=[RAIZ_COL], errors="ignore")
    P = P.loc[:, P.sum(axis=0) > 0]
    return P.div(P.sum(axis=1), axis=0)

## 4. Clustering sobre el perfil completo de hojas (sólo BAJA+2)

`clusterizar(...)` aplica **KMeans** sobre la matriz de perfiles `P` (sólo de los clientes BAJA+2). La distancia entre dos clientes en este espacio es la distancia entre sus perfiles de hojas: qué tan parecido los "trató" el bosque. KMeans simplemente corta ese espacio en `k` grupos.

Un detalle importante: los labels que devuelve KMeans son arbitrarios (el cluster "0" no significa nada especial), así que se **reordenan** de mayor a menor tamaño y se renumeran empezando en 1 (`cluster_1` es siempre el más grande). Esto hace que los resultados sean más fáciles de comparar entre corridas.

In [6]:
def clusterizar(P: pd.DataFrame, k: int, seed: int) -> pd.Series:
    """La distancia entre dos clientes es la distancia entre sus perfiles de hojas:
    qué tan parecido los trató el bosque. KMeans sólo corta ese espacio en k."""
    km = KMeans(n_clusters=k, n_init=20, random_state=seed).fit(P.values)
    orden = pd.Series(km.labels_).value_counts().index.tolist()  # cluster_1 = el más grande
    remap = {viejo: nuevo + 1 for nuevo, viejo in enumerate(orden)}
    return pd.Series([remap[l] for l in km.labels_], index=P.index, name=CLUSTER_COL)

## 5. Caracterización: share y lift por cluster; tendencias mensuales

 `caracterizar_clusters(...)`: para cada cluster, calcula el **lift** de cada atributo = `share_cluster / share_global` (qué tan sobre-representado está ese atributo en ese cluster respecto al promedio general). Se queda con los `top_n` atributos de mayor lift por cluster (filtrando primero los que tienen un `share_cluster` mínimo, para no destacar atributos raros que casi no aparecen). Estos son los atributos "definitorios" de cada cluster.

 `tendencias_por_cluster(...)`: para cada combinación (cluster, mes) y cada atributo, calcula un centro (media o mediana, según la banda elegida) y una banda de dispersión (IC95, desvío estándar o rango intercuartil), además de `n` = cantidad de clientes del cluster presentes ese mes. La banda se recorta al rango observado real del atributo para que el gráfico no muestre valores imposibles.

 `orden_atributos(...)`: define el orden en que van a aparecer las páginas del PDF — primero los atributos definitorios (agrupados por cluster y rank), después el resto en orden alfabético.

In [7]:
def caracterizar_clusters(P: pd.DataFrame, labels: pd.Series, top_n: int, share_min: float) -> pd.DataFrame:
    """Top-n atributos por cluster según lift = share_cluster / share_global."""
    share_global = P.mean(axis=0)
    filas = []
    for c, Pc in P.groupby(labels):
        share_c = Pc.mean(axis=0)
        tabla = pd.DataFrame({"share_cluster": share_c, "share_global": share_global,
                              "lift": share_c / share_global.replace(0, np.nan)})
        tabla = tabla[tabla["share_cluster"] >= share_min].sort_values("lift", ascending=False).head(top_n)
        for rank, (attr, r) in enumerate(tabla.iterrows(), start=1):
            filas.append({CLUSTER_COL: c, "rank": rank, "atributo": attr, **r.to_dict()})
    return pd.DataFrame(filas)


def tendencias_por_cluster(df_churn: pd.DataFrame, atributos: list[str], meses: list[int], banda: str) -> pd.DataFrame:
    """Por (cluster, mes): centro, banda y n de clientes presentes, por atributo.
    La banda se recorta al rango observado del atributo."""
    d = df_churn[df_churn[MES_COL].isin(meses)]
    g = d.groupby([CLUSTER_COL, MES_COL])[atributos]
    n = g.count()
    if banda == "iqr":
        centro, lo, hi = g.median(), g.quantile(0.25), g.quantile(0.75)
    else:
        centro, sd = g.mean(), g.std()
        ancho = sd if banda == "desvio" else 1.96 * sd / np.sqrt(n)
        lo, hi = centro - ancho, centro + ancho
    lo = lo.clip(lower=d[atributos].min(), axis=1)
    hi = hi.clip(upper=d[atributos].max(), axis=1)
    return pd.concat({"centro": centro, "lo": lo, "hi": hi, "n": n}, axis=1)


def orden_atributos(atributos_def: pd.DataFrame, feats: list[str]) -> list[str]:
    """Primero los definitorios (en orden de cluster y rank), después el resto por abecedario."""
    primero = atributos_def.sort_values([CLUSTER_COL, "rank"])["atributo"].unique().tolist()
    return primero + sorted((f for f in feats if f not in primero), key=str.lower)

## 6. PDF: página de resumen + una página apaisada por atributo

Esta sección genera el PDF final con matplotlib.

`_estilo(ax)`: aplica un estilo visual consistente a los ejes (sin bordes arriba/derecha, grilla suave, colores de tinta definidos en las constantes).

`_pagina_resumen(pdf, ctx)`: la primera página del PDF, con texto explicando el tamaño de la muestra, los hiperparámetros usados, el accuracy OOB del Random Forest, el tamaño de cada cluster y una guía de cómo leer el resto de las páginas.\n- `pdf_tendencias(...)`: genera una página por atributo (en el orden calculado por `orden_atributos`). Cada página tiene:

 - Un gráfico de líneas con la tendencia mensual del atributo por cluster (línea = centro, banda sombreada = dispersión).

 - Un texto que indica si el atributo es definitorio de algún cluster (y con qué lift/rank) o, si no lo es, cuál es el cluster donde tiene mayor lift.

 - Una tabla al pie con la cantidad de clientes de cada cluster presentes en cada mes, para visualizar cómo se van dando de baja.

In [8]:
def _estilo(ax) -> None:
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.tick_params(colors=INK_2, labelsize=9)
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


def _pagina_resumen(pdf: PdfPages, ctx: dict) -> None:
    fig = plt.figure(figsize=(11.69, 8.27))
    texto = [
        f"Muestra: {ctx['n_churn']:,} clientes BAJA+2 (toda su historia) + {ctx['n_fieles']:,} fieles.",
        f"Random Forest: {ctx['n_trees']} árboles, min_samples_leaf={ctx['min_samples_leaf']}, OOB accuracy {ctx['oob']:.3f}.",
        f"KMeans k={ctx['k']} sobre la matriz id × atributo (proporción de hojas definidas por cada atributo).",
        "",
        "Tamaño de cada cluster: " + ", ".join(f"cluster_{c}: {n:,}" for c, n in ctx["tam"].items()),
        "",
        "Cómo leer cada página:",
        "  • Título = atributo. Primero los definitorios (top-n por lift de cada cluster, en orden de cluster),",
        "    después todos los demás en orden alfabético.",
        f"  • Eje X = mes calendario ({ctx['meses'][0]}–{ctx['meses'][-1]}); {ctx['mes_excluido']} se excluye porque ya no hay BAJA+2 con historia ahí.",
        f"  • Línea = {ctx['centro']} del atributo entre los clientes del cluster presentes ese mes;",
        f"    banda = {ctx['banda_desc']}, recortada al rango observado del atributo.",
        "  • Los clusters se achican mes a mes (tabla de n al pie): los clientes se van dando de baja.",
        "  • Si un atributo define a un cluster, su línea debería despegarse claramente de las otras.",
    ]
    fig.text(0.06, 0.92, "Clusters de clientes BAJA+2 — tendencia mensual por atributo",
             fontsize=18, color=INK, weight="bold", va="top")
    fig.text(0.06, 0.84, "\n".join(texto), fontsize=11, color=INK_2, va="top", linespacing=1.6)
    pdf.savefig(fig)
    plt.close(fig)


def pdf_tendencias(tend: pd.DataFrame, atributos: list[str], atributos_def: pd.DataFrame, lift_all: pd.DataFrame,
                   meses: list[int], path: Path, ctx: dict) -> None:
    clusters = sorted(tend.index.get_level_values(CLUSTER_COL).unique())
    with PdfPages(path) as pdf:
        _pagina_resumen(pdf, ctx)
        for attr in atributos:
            define = atributos_def[atributos_def["atributo"] == attr].sort_values("lift", ascending=False)
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_axes([0.07, 0.30, 0.90, 0.54])
            x = np.arange(len(meses))
            for i, c in enumerate(clusters):
                s = tend.xs(c, level=CLUSTER_COL)
                centro = s["centro"][attr].reindex(meses).values
                lo, hi = s["lo"][attr].reindex(meses).values, s["hi"][attr].reindex(meses).values
                ax.fill_between(x, lo, hi, color=COLORES_CLUSTER[i], alpha=0.12, linewidth=0)
                ax.plot(x, centro, color=COLORES_CLUSTER[i], linewidth=2.2, marker="o", markersize=6, label=f"cluster_{c}")
            ax.set_xticks(x, meses)
            ax.set_xlim(-0.3, len(meses) - 0.7)
            _estilo(ax)
            ax.legend(frameon=False, fontsize=9, loc="upper left")
            ax.set_ylabel(attr, color=INK_2)

            fig.text(0.07, 0.945, attr, fontsize=20, color=INK, weight="bold")
            if len(define):
                quien = "; ".join(f"cluster_{r[CLUSTER_COL]} (lift {r['lift']:.2f}, rank {r['rank']})" for _, r in define.iterrows())
                rol = f"Define a {quien}."
            elif attr in lift_all.columns:
                c_max = lift_all[attr].idxmax()
                rol = f"No es definitorio de ningún cluster (mayor lift: cluster_{c_max}, {lift_all.loc[c_max, attr]:.2f})."
            else:
                rol = "Nunca definió una hoja del bosque."
            fig.text(0.07, 0.925, f"{rol}\nLínea = {ctx['centro']} mensual entre los clientes del cluster presentes ese mes; "
                     f"banda = {ctx['banda_desc']}, recortada al rango observado.", fontsize=10, color=INK_2, va="top", linespacing=1.4)

            # tabla de n por cluster y mes: hace visible el achicamiento de los grupos
            n_tab = tend["n"][attr].unstack(MES_COL).reindex(index=clusters, columns=meses).fillna(0).astype(int)
            ax_t = fig.add_axes([0.07, 0.05, 0.90, 0.17])
            ax_t.axis("off")
            tabla = ax_t.table(cellText=n_tab.values, rowLabels=[f"cluster_{c}" for c in clusters],
                               colLabels=[str(m) for m in meses], loc="center", cellLoc="center")
            tabla.auto_set_font_size(False)
            tabla.set_fontsize(8.5)
            tabla.scale(1, 1.15)
            for (r, _c), cell in tabla.get_celld().items():
                cell.set_edgecolor(GRID)
                cell.get_text().set_color(INK_2 if r == 0 else INK)
            ax_t.set_title("n de clientes del cluster presentes en cada mes", loc="left", fontsize=9, color=INK_2)
            pdf.savefig(fig)
            plt.close(fig)

## main

`parse_args()` define las opciones de línea de comandos con `argparse` (con los defaults documentados al principio del notebook).

`main()` orquesta todo el pipeline en orden:

1. Valida argumentos (`k` máximo y que el CSV exista).
2. Carga la muestra y calcula las features.
3. Entrena el Random Forest.
4. Cuenta hojas por cliente y normaliza a proporciones.
5. Clusteriza sólo los clientes BAJA+2.
6. Caracteriza los clusters (atributos definitorios + lift).
7. Calcula las tendencias mensuales por cluster.
8. Arma el diccionario `ctx` con todo el contexto para los textos del PDF.
9. Genera el PDF con `pdf_tendencias(...)`.


In [15]:
def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--csv", type=Path, default=Path("competencia_01_ct_julia.csv"))
    p.add_argument("--out", type=Path, default=Path("clusters_tendencias.pdf"))
    p.add_argument("--n-trees", type=int, default=300)
    p.add_argument("--min-samples-leaf", type=int, default=50)
    p.add_argument("--k", type=int, default=5)
    p.add_argument("--top-n", type=int, default=3)
    p.add_argument("--seed", type=int, default=214363)
    p.add_argument("--banda", choices=list(BANDAS), default="ic95")
    return p.parse_args()


def main():
    args = parse_args()
    if args.k > len(COLORES_CLUSTER):
        raise SystemExit(f"--k máximo {len(COLORES_CLUSTER)}")
    if not args.csv.exists():
        raise SystemExit(f"no encuentro {args.csv}")

    log(f"leyendo {args.csv.name}")
    df = cargar_muestra(args.csv, args.seed)
    meses_total = sorted(df[MES_COL].unique().tolist())
    meses = meses_total[:-1]  # el último mes no tiene BAJA+2 con historia
    feats = columnas_features(df)
    por_grupo = df.groupby(GRUPO_COL)[ID_COL].agg(ids="nunique", filas="size")
    log(f"muestra: {len(df):,} filas × {len(feats)} features · "
        + " · ".join(f"grupo {g}: {r.ids:,} ids / {r.filas:,} filas" for g, r in por_grupo.iterrows()))

    X = df[feats].to_numpy(dtype=np.float32)
    y = df[GRUPO_COL].to_numpy()
    ids = df[ID_COL].to_numpy()

    log(f"entrenando RF ({args.n_trees} árboles, min_samples_leaf={args.min_samples_leaf})")
    rf = entrenar_rf(X, y, args.n_trees, args.min_samples_leaf, args.seed)
    log(f"OOB accuracy {rf.oob_score_:.4f}")

    log("contando hojas por cliente")
    P = normalizar(contar_hojas_por_id(rf, X, ids, feats))
    ids_churn = pd.Index(df.loc[df[GRUPO_COL] == 1, ID_COL].unique())
    P_churn = P.loc[ids_churn]

    labels = clusterizar(P_churn, args.k, args.seed)
    tam = labels.value_counts().sort_index()
    log("clusters: " + ", ".join(f"cluster_{c}={n:,}" for c, n in tam.items()))

    attrs_def = caracterizar_clusters(P_churn, labels, args.top_n, share_min=0.5 / P_churn.shape[1])
    for c, g in attrs_def.groupby(CLUSTER_COL):
        log(f"  cluster_{c}: " + ", ".join(f"{r.atributo} (lift {r.lift:.1f})" for r in g.itertuples()))
    lift_all = P_churn.groupby(labels).mean() / P_churn.mean(axis=0)

    df_churn = df[df[GRUPO_COL] == 1].merge(labels, left_on=ID_COL, right_index=True)
    atributos_pdf = orden_atributos(attrs_def, feats)
    tend = tendencias_por_cluster(df_churn, atributos_pdf, meses, args.banda)

    ctx = dict(
        n_trees=args.n_trees, min_samples_leaf=args.min_samples_leaf, k=args.k, oob=rf.oob_score_,
        n_churn=int(por_grupo.loc[1, "ids"]), n_fieles=int(por_grupo.loc[0, "ids"]),
        meses=meses, mes_excluido=meses_total[-1], tam=tam.to_dict(),
        centro=BANDAS[args.banda][0], banda_desc=BANDAS[args.banda][1],
    )
    log(f"generando PDF ({len(atributos_pdf) + 1} páginas)")
    pdf_tendencias(tend, atributos_pdf, attrs_def, lift_all, meses, args.out, ctx)
    log(f"listo → {args.out}")

    return labels, P_churn, ids_churn

## Ejecución

El script original usa `argparse` leyendo `sys.argv`, pensado para correr desde la terminal. Para poder ejecutar `main()` tal cual desde el notebook, alcanza con **simular** los argumentos de línea de comandos asignando una lista a `sys.argv` antes de llamar a `main()`. Esto no modifica ninguna función del script: `parse_args()` sigue haciendo exactamente lo mismo (leer `sys.argv`).

Editá la ruta del `--csv` y las demás opciones según corresponda, y después ejecutá la celda. El PDF se va a generar en la ruta indicada por `--out`.

In [16]:
import sys
import os

BASE_URL = os.path.join('/mnt/', 'c')
BASE_DATA_URL = os.path.join('/mnt/', 'e')
COMP_URL = os.path.join(BASE_URL, 'Users', 'marco', 'Desktop', 'MASTER', 'MASTER', 'DMEyF', 'COMP1')
DATA_FOLDER = os.path.join(BASE_DATA_URL, 'DATASETS', 'DMEyF')
DATA_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria.csv')

sys.argv = [
    "z501_cluster_rf.py",
    "--csv", DATA_URL,
    "--out", "clusters_tendencias.pdf",
    # agregá acá más opciones si querés cambiar defaults, p.ej.:
    # "--n-trees", "300",
    # "--min-samples-leaf", "50",
    # "--k", "5",
    # "--top-n", "3",
    # "--seed", "214363",
    # "--banda", "ic95",
]

labels, P_churn, ids = main()

[ 179.7s] leyendo competencia_01_ternaria.csv
[ 194.6s] muestra: 38,713 filas × 153 features · grupo 0: 4,067 ids / 24,402 filas · grupo 1: 4,067 ids / 14,311 filas
[ 194.6s] entrenando RF (300 árboles, min_samples_leaf=50)
[ 198.7s] OOB accuracy 0.8541
[ 198.7s] contando hojas por cliente
[ 199.1s] clusters: cluster_1=1,241, cluster_2=1,176, cluster_3=865, cluster_4=566, cluster_5=219
[ 199.2s]   cluster_1: mttarjeta_visa_debitos_automaticos (lift 2.0), mtarjeta_visa_consumo (lift 2.0), ctarjeta_visa_debitos_automaticos (lift 1.9)
[ 199.2s]   cluster_2: mpagomiscuentas (lift 2.9), cpagomiscuentas (lift 2.9), mpayroll (lift 2.9)
[ 199.2s]   cluster_3: active_quarter (lift 2.3), Visa_mfinanciacion_limite (lift 1.8), Visa_mlimitecompra (lift 1.8)
[ 199.2s]   cluster_4: cdescubierto_preacordado (lift 4.5), internet (lift 4.2), active_quarter (lift 3.0)
[ 199.2s]   cluster_5: mprestamos_personales (lift 9.1), cprestamos_personales (lift 8.0), internet (lift 4.0)
[ 199.2s] generando PDF (15

In [17]:
P_churn.shape, labels.shape, ids.shape

((4067, 129), (4067,), (4067,))

In [77]:
kmeans = KMeans(
    n_clusters=6,
    n_init=20,
    random_state=230047
)
labels_mine = kmeans.fit_predict(P_churn)

In [78]:
pd.Series(labels_mine).value_counts()

5    1096
1     958
3     657
0     623
4     516
2     217
Name: count, dtype: int64

In [81]:
print("Inercia:", kmeans.inertia_)                          # menor
print("Iteraciones:", kmeans.n_iter_)
print("Silhouette:", silhouette_score(P_churn, labels_mine)) # mayor
print("Davies-Bouldin:", davies_bouldin_score(P_churn, labels_mine))  # menor
print("Calinski-Harabasz:", calinski_harabasz_score(P_churn, labels_mine))  # mayor

Inercia: 18.633736591510043
Iteraciones: 22
Silhouette: 0.12096761196495043
Davies-Bouldin: 2.2420749072173103
Calinski-Harabasz: 625.9865507390064


In [65]:
hier_model = AgglomerativeClustering(
    n_clusters=4,
    linkage='average',
    # linkage='ward',
    # metric='cosine'
    metric='euclidean'
)
labels_hier = hier_model.fit_predict(P_churn)


In [66]:
pd.Series(labels_hier).value_counts()

1    3097
0     952
3      12
2       6
Name: count, dtype: int64

In [62]:
silhouette_score(P_churn, labels_hier),

(0.20641425034195007,)

In [14]:
# P_churn['label']=labels